# Self Attention: 1 head.

In [3]:
import math 
import torch 
import torch.nn as nn 

class SelfAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim 
        self.Q_proj = nn.Linear(dim, dim)
        self.K_proj = nn.Linear(dim, dim)
        self.V_proj = nn.Linear(dim, dim)
        self.nn_dropout = nn.Dropout(0.1) 
        self.O_proj = nn.Linear(dim, dim)
        # (b, s, h)
    
    def forward(self, x, att_mask = None):
        Q = self.Q_proj(x)
        K = self.K_proj(x)
        V = self.V_proj(x)
        att_weight = Q @ K.transpose(-2, -1) / math.sqrt(self.dim)


        # 输入 X维度：(batch_size, seq_len, hidden_dim)
        # 经过线性变换后，Q、K、V的维度均为 (batch_size, seq_len, hidden_dim)
        # K.transpose(-2, -1)表示对张量 K的​​倒数第二维和倒数第一维进行转置​​：
        # -1表示最后一个维度（hidden_dim）
        # -2表示倒数第二个维度（seq_len）
        # 转置后 K的维度变为：(batch_size, hidden_dim, seq_len)
        # 此时 Q @ K^T = (b, s, h) @ (b, h, s) = (b, s, s)
        
        if att_mask is not None:
            att_weight = att_weight.masked_fill(att_mask == 0, float('-inf'))
        att_weight = torch.softmax(att_weight, dim=-1)
        # 经过 Mask 后的 Q @ K^T, (b, s, s)
        print(att_weight)
        att_weight = self.nn_dropout(att_weight)
        output = att_weight @ V
        # (b, s, s) @ (b, s, h) = (b, s, h)
        ret = self.O_proj(output)
        #  (b, s, h) * (h, h) = (b, s, h)
        return ret
        

X = torch.rand(3, 4, 2)
b = torch.tensor(
    [
        [1, 1, 1, 0],
        [1, 1, 0, 0],
        [1, 0, 0, 0],
    ]
)
print(b.shape)
mask = b.unsqueeze(dim=1).repeat(1, 4, 1)

net = SelfAttention(2)
net(X, mask).shape

torch.Size([3, 4])
tensor([[[0.3294, 0.3405, 0.3301, 0.0000],
         [0.3291, 0.3407, 0.3302, 0.0000],
         [0.3280, 0.3426, 0.3294, 0.0000],
         [0.3279, 0.3432, 0.3289, 0.0000]],

        [[0.5067, 0.4933, 0.0000, 0.0000],
         [0.5076, 0.4924, 0.0000, 0.0000],
         [0.5080, 0.4920, 0.0000, 0.0000],
         [0.5064, 0.4936, 0.0000, 0.0000]],

        [[1.0000, 0.0000, 0.0000, 0.0000],
         [1.0000, 0.0000, 0.0000, 0.0000],
         [1.0000, 0.0000, 0.0000, 0.0000],
         [1.0000, 0.0000, 0.0000, 0.0000]]], grad_fn=<SoftmaxBackward0>)


torch.Size([3, 4, 2])

# Attention: Multi Head Attention

In [ ]:
import math 
import torch 
import torch.nn as nn 

class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_dim, head_num, dropout = 0.1):
        super().__init__()
        self.hidden_dim = hidden_dim 
        self.head_num = head_num 
        self.head_dim = hidden_dim // head_num
        self.Q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.K_proj = nn.Linear(hidden_dim, hidden_dim)
        self.V_proj = nn.Linear(hidden_dim, hidden_dim)
        self.O_proj = nn.Linear(hidden_dim, hidden_dim)
        self.nn_dropout = nn.Dropout(dropout)

    def forward(self, x, att_mask = None ):
        batch_size, seq_len, _ = x.size()
        Q = self.Q_proj(x)
        K = self.Q_proj(x)
        V = self.Q_proj(x)
        # (b, s, head, h) -> (b, head, s, h)
        Q_state = Q.view(batch_size, seq_len, self.head_num, self.head_dim).transpose(1, 2)
        K_state = K.view(batch_size, seq_len, self.head_num, self.head_dim).transpose(1, 2)
        V_state = V.view(batch_size, seq_len, self.head_num, self.head_dim).transpose(1, 2)
        
        att_score = torch.matmul(
            # (b, head, s, h) @ (b, head, h, s) = (b, head, s, s)
            Q_state, K_state.transpose(-1, -2)
        ) / math.sqrt(self.head_dim)
        
        if att_mask is not None:
            att_score = att_score.masked_fill(att_mask == 0, float('-inf'))


        att_score = torch.softmax(att_score, dim = -1)
        att_score = self.nn_dropout(att_score)
        # (b, head, s, s) @ (b, head, s, h) = (b, head, s, h)
        output_mid = att_score @ V_state
        # (b, head, s, h) -> (b, s, head, h)
        output_mid = output_mid.transpose(1, 2).contiguous()
        # (b, s, head, h) -> (b, s, head * h)
        output = output_mid.view(batch_size, seq_len, -1)
        output = self.O_proj(output)
        return output


attention_mask = (
    torch.tensor(
        [
            [0, 1],
            [0, 0],
            [1, 0],
        ]
    )
    .unsqueeze(1)
    .unsqueeze(2)
    .expand(3, 8, 2, 2)
)

x = torch.rand(3, 2, 128)
net = MultiHeadAttention(128, 8)
net(x, attention_mask).shape

torch.Size([3, 2, 128])